# Chapter 2: Agent Cognitive Architecture
## Memory, Context, and Reasoning - Code Examples

This notebook contains practical implementations of the concepts covered in Chapter 2:
- Multi-tier memory systems (short-term, long-term, episodic)
- Context engineering and management
- Meta-reasoning and self-reflection
- Tree of Thoughts implementation

### Setup

Install dependencies using `uv`:

```bash
uv add openai
uv add anthropic
uv add langchain
uv add langchain-openai
uv add langgraph
uv add chromadb
uv add tiktoken
uv add python-dotenv
```

In [ ]:
# Import required libraries
import os
from dotenv import load_dotenv
from typing import List, Dict, Any, Optional
import json
from datetime import datetime
import tiktoken

# Load environment variables
load_dotenv()

# Verify API keys
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found"
print("Environment setup complete")

## Part 1: Implementing Multi-Tier Memory Architecture

We'll build a complete memory system with short-term, long-term, and episodic memory.

In [ ]:
from openai import OpenAI
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class ShortTermMemory:
    """Manages the agent's working memory (context window)."""
    
    def __init__(self, max_tokens: int = 4000):
        self.messages = []
        self.max_tokens = max_tokens
        self.encoding = tiktoken.encoding_for_model("gpt-4")
        
    def add_message(self, role: str, content: str):
        """Add a message to short-term memory."""
        self.messages.append({"role": role, "content": content})
        self._trim_if_needed()
        
    def _count_tokens(self, messages: List[Dict]) -> int:
        """Count tokens in messages."""
        total = 0
        for msg in messages:
            total += len(self.encoding.encode(msg["content"]))
        return total
    
    def _trim_if_needed(self):
        """Remove old messages if approaching token limit."""
        while self._count_tokens(self.messages) > self.max_tokens and len(self.messages) > 1:
            # Keep system message, remove oldest user/assistant message
            if self.messages[0]["role"] == "system":
                self.messages.pop(1)
            else:
                self.messages.pop(0)
    
    def get_messages(self) -> List[Dict]:
        """Retrieve all messages in short-term memory."""
        return self.messages
    
    def clear(self):
        """Clear short-term memory (end session)."""
        self.messages = []


class LongTermMemory:
    """Manages persistent semantic memory using embeddings."""
    
    def __init__(self, client: OpenAI):
        self.client = client
        self.memories = []  # In production, use a vector database
        
    def store(self, content: str, metadata: Optional[Dict] = None):
        """Store information in long-term memory."""
        # Generate embedding
        embedding = self._get_embedding(content)
        
        # Store with metadata
        memory = {
            "content": content,
            "embedding": embedding,
            "metadata": metadata or {},
            "timestamp": datetime.now().isoformat()
        }
        self.memories.append(memory)
        
    def retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
        """Retrieve relevant memories using semantic search."""
        if not self.memories:
            return []
            
        # Get query embedding
        query_embedding = self._get_embedding(query)
        
        # Calculate similarities
        similarities = []
        for memory in self.memories:
            similarity = cosine_similarity(
                [query_embedding],
                [memory["embedding"]]
            )[0][0]
            similarities.append((similarity, memory))
        
        # Sort by similarity and return top-k
        similarities.sort(reverse=True, key=lambda x: x[0])
        return [mem for _, mem in similarities[:top_k]]
    
    def _get_embedding(self, text: str) -> List[float]:
        """Generate embedding for text."""
        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding


class EpisodicMemory:
    """Manages memory of specific experiences and events."""
    
    def __init__(self, client: OpenAI):
        self.client = client
        self.episodes = []
        
    def record_episode(self, context: str, action: str, outcome: str, success: bool):
        """Record a significant episode."""
        episode_text = f"Context: {context}\nAction: {action}\nOutcome: {outcome}"
        embedding = self._get_embedding(episode_text)
        
        episode = {
            "context": context,
            "action": action,
            "outcome": outcome,
            "success": success,
            "embedding": embedding,
            "timestamp": datetime.now().isoformat()
        }
        self.episodes.append(episode)
        
    def find_similar_episodes(self, current_context: str, top_k: int = 3) -> List[Dict]:
        """Find past episodes similar to current situation."""
        if not self.episodes:
            return []
            
        context_embedding = self._get_embedding(current_context)
        
        # Calculate similarities
        similarities = []
        for episode in self.episodes:
            similarity = cosine_similarity(
                [context_embedding],
                [episode["embedding"]]
            )[0][0]
            similarities.append((similarity, episode))
        
        # Sort and return top-k
        similarities.sort(reverse=True, key=lambda x: x[0])
        return [ep for _, ep in similarities[:top_k]]
    
    def _get_embedding(self, text: str) -> List[float]:
        """Generate embedding for text."""
        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding


# Test the memory systems
client = OpenAI()

# Short-term memory
stm = ShortTermMemory(max_tokens=1000)
stm.add_message("system", "You are a helpful assistant.")
stm.add_message("user", "What is machine learning?")
stm.add_message("assistant", "Machine learning is a subset of AI...")
print(f"Short-term memory has {len(stm.get_messages())} messages")

# Long-term memory
ltm = LongTermMemory(client)
ltm.store("The user prefers Python over JavaScript", {"type": "preference"})
ltm.store("The user is working on a machine learning project", {"type": "context"})
relevant = ltm.retrieve("What programming language does the user like?")
print(f"\nRetrieved from long-term memory: {relevant[0]['content']}")

# Episodic memory
em = EpisodicMemory(client)
em.record_episode(
    context="User asked about data visualization",
    action="Recommended matplotlib and seaborn",
    outcome="User was satisfied",
    success=True
)
similar = em.find_similar_episodes("User asking about plotting libraries")
if similar:
    print(f"\nSimilar past episode: {similar[0]['action']}")

## Part 2: Context Engineering and Management

Implementing strategies to manage context window efficiently.

In [ ]:
class ContextManager:
    """Manages context window with compression and summarization."""
    
    def __init__(self, client: OpenAI, model_context_limit: int = 128000, pre_rot_threshold: float = 0.25):
        self.client = client
        self.context_limit = model_context_limit
        self.threshold = int(model_context_limit * pre_rot_threshold)
        self.encoding = tiktoken.encoding_for_model("gpt-4")
        
    def count_tokens(self, messages: List[Dict]) -> int:
        """Count total tokens in messages."""
        total = 0
        for msg in messages:
            total += len(self.encoding.encode(msg["content"]))
        return total
    
    def should_compress(self, messages: List[Dict]) -> bool:
        """Check if context should be compressed."""
        return self.count_tokens(messages) >= self.threshold
    
    def compress_context(self, messages: List[Dict], keep_recent: int = 10) -> List[Dict]:
        """Compress old messages while preserving recent context."""
        if not self.should_compress(messages):
            return messages
            
        # Keep system message and recent messages
        system_msg = [messages[0]] if messages[0]["role"] == "system" else []
        recent_msgs = messages[-keep_recent:]
        
        # Summarize middle messages
        start_idx = 1 if system_msg else 0
        end_idx = -keep_recent
        old_msgs = messages[start_idx:end_idx]
        
        if old_msgs:
            summary = self._summarize_messages(old_msgs)
            summary_msg = {
                "role": "system",
                "content": f"Previous conversation summary: {summary}"
            }
            return system_msg + [summary_msg] + recent_msgs
        
        return messages
    
    def _summarize_messages(self, messages: List[Dict]) -> str:
        """Summarize a list of messages."""
        conversation = "\n".join(
            f"{msg['role']}: {msg['content']}" for msg in messages
        )
        
        response = self.client.chat.completions.create(
            model="gpt-4",
            messages=[
                {
                    "role": "system",
                    "content": "Summarize the following conversation concisely, preserving key facts and context."
                },
                {
                    "role": "user",
                    "content": conversation
                }
            ],
            temperature=0
        )
        
        return response.choices[0].message.content


# Test context management
context_mgr = ContextManager(client)

# Create a conversation that exceeds threshold
test_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell me about Python."},
    {"role": "assistant", "content": "Python is a high-level programming language..." * 100},
    {"role": "user", "content": "What about Java?"},
    {"role": "assistant", "content": "Java is an object-oriented language..." * 100},
]

token_count = context_mgr.count_tokens(test_messages)
print(f"Token count: {token_count}")
print(f"Should compress: {context_mgr.should_compress(test_messages)}")

if context_mgr.should_compress(test_messages):
    compressed = context_mgr.compress_context(test_messages)
    new_count = context_mgr.count_tokens(compressed)
    print(f"Compressed to: {new_count} tokens")
    print(f"Reduction: {((token_count - new_count) / token_count * 100):.1f}%")

## Part 3: Reflexion - Meta-Reasoning and Self-Reflection

Implementing self-evaluation and reflection loops.

In [ ]:
class ReflexionAgent:
    """Agent with self-reflection and learning capabilities."""
    
    def __init__(self, client: OpenAI, model: str = "gpt-4"):
        self.client = client
        self.model = model
        self.reflections = []  # Store learned insights
        
    def solve(self, problem: str, max_iterations: int = 3) -> Dict:
        """Solve problem with reflection loop."""
        for iteration in range(max_iterations):
            print(f"\n=== Iteration {iteration + 1} ===")
            
            # Generate solution
            solution = self._generate_solution(problem)
            print(f"Solution: {solution[:200]}...")
            
            # Evaluate solution
            evaluation = self._evaluate_solution(problem, solution)
            print(f"Evaluation: {evaluation['verdict']} (score: {evaluation['score']})")
            
            if evaluation["score"] >= 0.8:  # Success threshold
                return {
                    "solution": solution,
                    "iterations": iteration + 1,
                    "success": True
                }
            
            # Reflect on failure
            reflection = self._reflect(problem, solution, evaluation)
            self.reflections.append(reflection)
            print(f"Reflection: {reflection[:200]}...")
        
        return {
            "solution": solution,
            "iterations": max_iterations,
            "success": False
        }
    
    def _generate_solution(self, problem: str) -> str:
        """Generate solution considering past reflections."""
        # Include past reflections as context
        context = ""
        if self.reflections:
            context = "Past reflections and learnings:\n"
            context += "\n".join(f"- {r}" for r in self.reflections[-3:])
            context += "\n\nUse these insights to improve your solution.\n\n"
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": f"{context}Solve the following problem step by step."
                },
                {
                    "role": "user",
                    "content": problem
                }
            ],
            temperature=0.7
        )
        
        return response.choices[0].message.content
    
    def _evaluate_solution(self, problem: str, solution: str) -> Dict:
        """Evaluate the quality of the solution."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": """Evaluate the solution on a scale of 0.0 to 1.0.
                    Return JSON with: {"score": float, "verdict": str, "feedback": str}"""
                },
                {
                    "role": "user",
                    "content": f"Problem: {problem}\n\nSolution: {solution}"
                }
            ],
            temperature=0
        )
        
        try:
            return json.loads(response.choices[0].message.content)
        except:
            return {"score": 0.5, "verdict": "uncertain", "feedback": "Could not parse"}
    
    def _reflect(self, problem: str, solution: str, evaluation: Dict) -> str:
        """Generate reflection on failed attempt."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": "Reflect on what went wrong and what to improve."
                },
                {
                    "role": "user",
                    "content": f"""Problem: {problem}
                    
Your solution: {solution}

Evaluation: {evaluation['feedback']}

Reflect on what went wrong and how to improve:"""
                }
            ],
            temperature=0.7
        )
        
        return response.choices[0].message.content


# Test Reflexion agent
agent = ReflexionAgent(client)

problem = """Write a Python function that finds the longest palindromic substring
in a given string. The function should be efficient and handle edge cases."""

result = agent.solve(problem, max_iterations=2)
print(f"\n=== Final Result ===")
print(f"Success: {result['success']}")
print(f"Iterations: {result['iterations']}")

## Part 4: Tree of Thoughts Implementation

Implementing ToT for exploring multiple reasoning paths.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class ThoughtNode:
    """Represents a thought in the tree."""
    content: str
    depth: int
    score: float = 0.0
    parent: Optional['ThoughtNode'] = None
    

class TreeOfThoughts:
    """Tree of Thoughts reasoning implementation."""
    
    def __init__(self, client: OpenAI, max_depth: int = 3, branching_factor: int = 3, model: str = "gpt-4"):
        self.client = client
        self.max_depth = max_depth
        self.branching_factor = branching_factor
        self.model = model
        
    def solve(self, problem: str) -> Dict:
        """Solve problem using Tree of Thoughts."""
        root = ThoughtNode(content=problem, depth=0)
        solution = self._search(root)
        return solution
    
    def _search(self, node: ThoughtNode, path: List[str] = None) -> Dict:
        """Depth-first search through thought tree."""
        if path is None:
            path = []
            
        path = path + [node.content]
        
        if node.depth >= self.max_depth:
            # Reached max depth, evaluate final solution
            final_solution = self._synthesize_path(path)
            score = self._evaluate_solution(final_solution)
            return {"solution": final_solution, "score": score, "path": path}
        
        # Generate multiple next thoughts
        thoughts = self._generate_thoughts(node, k=self.branching_factor)
        
        # Evaluate and rank thoughts
        scored_thoughts = []
        for thought in thoughts:
            score = self._evaluate_thought(thought, path)
            scored_thoughts.append((score, thought))
        
        scored_thoughts.sort(reverse=True, key=lambda x: x[0])
        
        # Explore best branches
        best_solution = None
        best_score = -float('inf')
        
        for score, thought in scored_thoughts:
            if score < 0.3:  # Prune unpromising branches
                continue
            
            child = ThoughtNode(content=thought, depth=node.depth + 1, parent=node)
            result = self._search(child, path)
            
            if result["score"] > best_score:
                best_solution = result
                best_score = result["score"]
        
        return best_solution if best_solution else {"solution": "", "score": 0.0, "path": path}
    
    def _generate_thoughts(self, node: ThoughtNode, k: int) -> List[str]:
        """Generate k different next thoughts."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": f"""Generate {k} different approaches to continue solving this problem.
                    Each approach should be distinct and explore a different strategy.
                    Return only the approaches, one per line."""
                },
                {
                    "role": "user",
                    "content": f"Current problem state: {node.content}"
                }
            ],
            temperature=0.8
        )
        
        thoughts = response.choices[0].message.content.strip().split('\n')
        return [t.strip() for t in thoughts if t.strip()][:k]
    
    def _evaluate_thought(self, thought: str, path: List[str]) -> float:
        """Evaluate the promise of a thought."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": "Rate how promising this approach is on a scale of 0.0 to 1.0. Return only the number."
                },
                {
                    "role": "user",
                    "content": f"Path so far: {' -> '.join(path)}\nNext step: {thought}"
                }
            ],
            temperature=0
        )
        
        try:
            return float(response.choices[0].message.content.strip())
        except:
            return 0.5
    
    def _synthesize_path(self, path: List[str]) -> str:
        """Synthesize final solution from reasoning path."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": "Synthesize a complete solution from this reasoning path."
                },
                {
                    "role": "user",
                    "content": "Reasoning path:\n" + "\n".join(f"{i+1}. {p}" for i, p in enumerate(path))
                }
            ],
            temperature=0
        )
        
        return response.choices[0].message.content
    
    def _evaluate_solution(self, solution: str) -> float:
        """Evaluate final solution quality."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": "Rate this solution on a scale of 0.0 to 1.0. Return only the number."
                },
                {
                    "role": "user",
                    "content": solution
                }
            ],
            temperature=0
        )
        
        try:
            return float(response.choices[0].message.content.strip())
        except:
            return 0.5


# Test Tree of Thoughts
tot = TreeOfThoughts(client, max_depth=2, branching_factor=2)

problem = """Design a system for managing a library's book inventory.
Consider multiple approaches and choose the best one."""

print("Solving with Tree of Thoughts...\n")
result = tot.solve(problem)

print("\n=== Result ===")
print(f"Score: {result['score']}")
print(f"\nReasoning path:")
for i, step in enumerate(result['path'], 1):
    print(f"{i}. {step}")
print(f"\nFinal solution:\n{result['solution']}")

## Part 5: Complete Agent with Integrated Cognitive Architecture

Putting it all together into a production-ready agent.

In [ ]:
class CognitiveAgent:
    """Complete agent with multi-tier memory and advanced reasoning."""
    
    def __init__(self, client: OpenAI, model: str = "gpt-4"):
        self.client = client
        self.model = model
        
        # Memory systems
        self.short_term = ShortTermMemory(max_tokens=4000)
        self.long_term = LongTermMemory(client)
        self.episodic = EpisodicMemory(client)
        
        # Context management
        self.context_mgr = ContextManager(client)
        
        # Reasoning
        self.reflections = []
        
    def chat(self, user_message: str, use_reflection: bool = False) -> str:
        """Process user message with full cognitive architecture."""
        # 1. Retrieve relevant long-term memories
        relevant_memories = self.long_term.retrieve(user_message, top_k=3)
        
        # 2. Find similar past episodes
        similar_episodes = self.episodic.find_similar_episodes(user_message, top_k=2)
        
        # 3. Build context
        context_parts = []
        
        if relevant_memories:
            context_parts.append("Relevant context from memory:")
            for mem in relevant_memories:
                context_parts.append(f"- {mem['content']}")
        
        if similar_episodes:
            context_parts.append("\nSimilar past experiences:")
            for ep in similar_episodes:
                if ep['success']:
                    context_parts.append(f"- When {ep['context']}, {ep['action']} worked well")
        
        enriched_context = "\n".join(context_parts) if context_parts else ""
        
        # 4. Add to short-term memory
        if enriched_context:
            self.short_term.add_message("system", enriched_context)
        
        self.short_term.add_message("user", user_message)
        
        # 5. Compress if needed
        messages = self.short_term.get_messages()
        if self.context_mgr.should_compress(messages):
            messages = self.context_mgr.compress_context(messages)
        
        # 6. Generate response
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.7
        )
        
        assistant_message = response.choices[0].message.content
        
        # 7. Self-evaluation (optional)
        if use_reflection:
            quality = self._evaluate_response(user_message, assistant_message)
            if quality < 0.7:
                reflection = self._reflect(user_message, assistant_message, quality)
                self.reflections.append(reflection)
        
        # 8. Update memories
        self.short_term.add_message("assistant", assistant_message)
        
        # Extract and store important facts
        facts = self._extract_facts(user_message, assistant_message)
        for fact in facts:
            self.long_term.store(fact, {"source": "conversation"})
        
        return assistant_message
    
    def record_interaction(self, user_message: str, response: str, success: bool):
        """Record interaction in episodic memory."""
        self.episodic.record_episode(
            context=user_message,
            action=f"Responded with: {response[:100]}...",
            outcome="User satisfied" if success else "User not satisfied",
            success=success
        )
    
    def _evaluate_response(self, query: str, response: str) -> float:
        """Evaluate response quality."""
        eval_response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": "Rate the quality of this response on a scale of 0.0 to 1.0. Return only the number."
                },
                {
                    "role": "user",
                    "content": f"Query: {query}\nResponse: {response}"
                }
            ],
            temperature=0
        )
        
        try:
            return float(eval_response.choices[0].message.content.strip())
        except:
            return 0.5
    
    def _reflect(self, query: str, response: str, quality: float) -> str:
        """Generate reflection on response."""
        reflection_response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": "Reflect on how to improve this response."
                },
                {
                    "role": "user",
                    "content": f"Query: {query}\nResponse: {response}\nQuality: {quality}\n\nHow could this be improved?"
                }
            ],
            temperature=0.7
        )
        
        return reflection_response.choices[0].message.content
    
    def _extract_facts(self, query: str, response: str) -> List[str]:
        """Extract important facts to store in long-term memory."""
        extraction = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": "Extract key facts or preferences from this conversation. Return a JSON list of strings."
                },
                {
                    "role": "user",
                    "content": f"User: {query}\nAssistant: {response}"
                }
            ],
            temperature=0
        )
        
        try:
            return json.loads(extraction.choices[0].message.content)
        except:
            return []


# Test the complete cognitive agent
agent = CognitiveAgent(client)

# Store some background knowledge
agent.long_term.store("User is a Python developer interested in AI", {"type": "profile"})
agent.long_term.store("User prefers practical examples over theory", {"type": "preference"})

# Have a conversation
print("=== Conversation ===")
response1 = agent.chat("I want to learn about neural networks")
print(f"Agent: {response1}\n")

response2 = agent.chat("Can you show me a code example?")
print(f"Agent: {response2}\n")

# Record successful interaction
agent.record_interaction(
    "Can you show me a code example?",
    response2,
    success=True
)

print("\n=== Memory Stats ===")
print(f"Short-term memory: {len(agent.short_term.get_messages())} messages")
print(f"Long-term memory: {len(agent.long_term.memories)} facts")
print(f"Episodic memory: {len(agent.episodic.episodes)} episodes")

## Summary

In this notebook, we implemented:

1. **Multi-Tier Memory Architecture**:
   - Short-term memory with automatic trimming
   - Long-term semantic memory with vector embeddings
   - Episodic memory for learning from experience

2. **Context Engineering**:
   - Token counting and monitoring
   - Context compression with summarization
   - Pre-rot threshold management

3. **Reflexion Framework**:
   - Self-evaluation of solutions
   - Reflection loops for continuous improvement
   - Learning from mistakes

4. **Tree of Thoughts**:
   - Multiple reasoning path exploration
   - Branch evaluation and pruning
   - Path synthesis for final solutions

5. **Complete Cognitive Agent**:
   - Integration of all memory systems
   - Context-aware reasoning
   - Continuous learning and adaptation

### Key Takeaways:

- Memory systems enable agents to learn and improve over time
- Context engineering is critical for managing limited working memory
- Self-reflection dramatically improves agent performance
- Tree of Thoughts enables exploration of complex solution spaces
- Integration of multiple techniques creates more capable agents

### Next Steps:

- Integrate with production vector databases (Pinecone, Weaviate)
- Add persistence for memory systems across sessions
- Implement more sophisticated retrieval strategies
- Add observability and logging for production deployment
- Optimize for cost and latency